In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import h5py as hf

from dataclasses import dataclass

In [ ]:
@dataclass
class Experiment:
    loss: np.ndarray
    loss_err: np.ndarray
    entropy: np.ndarray
    entropy_err: np.ndarray
    trace: np.ndarray   
    trace_err: np.ndarray

In [ ]:
def load_data(experiment):
    """
    Load the data for a specific experiment
    """
    loss = []
    entropy = []
    trace = []
    for i in range(1, 21):
        try:
            with hf.File(f"{experiment}/train_recorder_{i}.h5", "r") as f:
                # if f["loss"][:].shape[0] == 5000 and f["entropy"][:].shape[0] == 5000 and f["trace"][:].shape[0] == 5000:
                loss.append(f["loss"][:5000])
                entropy.append(f["entropy"][:5000])
                trace.append(f["trace"][:5000])
        except FileNotFoundError:
            pass

    return Experiment(
        loss=np.mean(loss, axis=0),
        loss_err=np.std(loss, axis=0),
        entropy=np.mean(entropy, axis=0),
        entropy_err=np.std(entropy, axis=0),
        trace=np.mean(trace, axis=0),
        trace_err=np.std(trace, axis=0),
    )

In [ ]:
# experiments = [
#     "perceptron",
#     "ce-perceptron", 
#     "ce-single-layer-10", 
#     "ce-single-layer-100", 
#     "ce-two-layer-10", 
#     "ce-two-layer-100",
# ]
experiments = [
    "perceptron",
    "one-layer-10", 
    "one-layer-100", 
    "two-layer-10",
    "two-layer-100",
]

In [ ]:
experiments = {
    experiment: load_data(experiment) for experiment in experiments
    }

# Plots

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
titles = ["Loss", "Entropy", "Trace"]
colours = ["#F18F01", "#DA4167", "#EEF36A", "#483C46", "#3A1772"]

i = 0
j = 0
epochs = np.linspace(1, 5000, 5000)
for key, value in experiments.items():
    ax[i].plot(value.loss, c=colours[j], alpha=0.3)
    ax[i].plot(
        np.convolve(value.loss, np.ones(100) / 100, mode="valid"),
        label=key,
        c=colours[j]
        )
    # ax[i].fill_between(
    #     epochs, 
    #     value.loss - value.loss_err, 
    #     value.loss + value.loss_err, 
    #     alpha=0.4,
    #     color=colours[j]
    #     )
    ax[i].set_title(titles[i])
    ax[i].set_xlabel("Epochs")
    ax[i].set_ylabel("Loss")
    ax[i].set_yscale("log")
    ax[i].set_ylim(1e-3, 0.2)

    j += 1

i = 1
j = 0
for key, value in experiments.items():
    ax[i].plot(value.entropy, c=colours[j], alpha=0.3)
    ax[i].plot(
        np.convolve(value.entropy, np.ones(100) / 100, mode="valid"),
        c=colours[j]
        )
    # ax[i].fill_between(
    #     epochs,
    #     value.entropy - value.entropy_err, 
    #     value.entropy + value.entropy_err, 
    #     alpha=0.4,
    #     color=colours[j]
    #     )
    ax[i].set_title(titles[i])
    ax[i].set_xlabel("Epochs")
    ax[i].set_ylabel("Entropy")
    j += 1

i = 2
j = 0
for key, value in experiments.items():
    ax[i].plot(value.trace, alpha=0.3, c=colours[j])
    ax[i].plot(
        np.convolve(value.trace, np.ones(100) / 100, mode="valid"),
        c=colours[j]
        )
    # ax[i].fill_between(
    #     epochs, 
    #     value.trace - value.trace_err, 
    #     value.trace + value.trace_err, 
    #     alpha=0.4,
    #     color=colours[j]
    #     )
    ax[i].set_title(titles[i])
    ax[i].set_xlabel("Epochs")
    ax[i].set_ylabel("Trace")
    ax[i].set_yscale("log")
    j += 1

fig.legend(loc=(0.45, 0.4))
plt.savefig("without_error_analysis.png", dpi=800)
plt.show()